# 02 · Benchmark LLM labelers and measure agreement

Which model should label 20,000 reviews? This notebook labels the human-annotated holdout with
**6 models from 5 vendors** (GPT-4o-mini, GPT-4.1-mini, Claude Haiku 4.5, Grok-3-mini,
Gemini 2.5 Flash-Lite, DeepSeek-chat), three runs each, then compares them against the human labels.

* Same prompt for every model; JSON output parsed and validated; cost and latency tracked.
* Majority vote across the 3 runs, and run-to-run reliability (Krippendorff's alpha).
* Agreement with the human labels: Krippendorff's alpha, macro F1, AUC and MCC, per emotion and overall.
* A routed-ensemble experiment (best model per label) against the single best model.

**Needs:** the holdout from notebook 01, the human-labeled holdout file, and API keys for the vendors you enable
(`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `XAI_API_KEY`, `DEEPSEEK_API_KEY`).
Labeling calls cost real money, so trim `VENDORS` in the config cell to try it cheaply.

Helper functions here are adapted from the course instructor's starter pipeline (marked "Ringel" in comments).


In [ ]:
# ── Setup (runs locally or on Colab) ─────────────────────────────────────────
# Install dependencies first:  pip install -r requirements.txt
import os

# Where datasets, LLM labels, checkpoints and models are written (not committed to git).
# Colab: mount Drive and set CAPSTONE_DATA_DIR to a Drive folder to persist between sessions.
DATA_DIR = os.environ.get("CAPSTONE_DATA_DIR", "data")
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_DIR = DATA_DIR

# API keys: environment variables first, Colab Secrets as a fallback.
_ENV_NAMES = {
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "xai": "XAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "minimax": "MINIMAX_API_KEY",
}

def get_api_key(name):
    value = os.environ.get(_ENV_NAMES.get(name, name)) or os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata as _colab_userdata
        return _colab_userdata.get(name)
    except Exception:
        return None

class userdata:  # drop-in for google.colab.userdata
    get = staticmethod(get_api_key)


In [ ]:
import os, json, time, datetime
import numpy as np
import pandas as pd
import krippendorff
from tqdm import tqdm
from openai import OpenAI
import anthropic
from google import genai
from sklearn.metrics import (
    f1_score, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, hamming_loss, jaccard_score
)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = DATA_DIR
HOLDOUT_DIR    = f"{OUTPUT_DIR}/holdout_llm_labels"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
os.makedirs(HOLDOUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Labels ────────────────────────────────────────────────────────────────────
EMOTIONS = [
    "Joy", "Love", "Surprise", "Pride", "Gratitude",
    "Contentment", "Relief", "Hope", "Interest",
    "Anger", "Frustration", "Disappointment",
    "Sadness", "Fear", "Guilt", "Disgust"
]
EVAL_DIMS = {
    "Trust":          ["Low", "Medium", "High"],
    "Commitment":     ["Low", "Medium", "High"],
    "Recommendation": ["Would Not Recommend", "Neutral", "Would Recommend"],
    "Sentiment":      ["Negative", "Neutral", "Positive"],
}

# ── Vendor/model config (adapted from Ringel pipeline Cell 11) ────────────────
# All prices per 1M tokens USD
VENDORS = {
    "openai": {
        "models": {
            "gpt-4o-mini": {"supplier": "openai", "price_in": 0.15, "price_out": 0.60},
            "gpt-4.1-mini": {"supplier": "openai", "price_in": 0.40, "price_out": 1.60},
            }
        },
    "anthropic": {
        "models": {
            "claude-haiku-4-5-20251001": {"supplier": "anthropic", "price_in": 1.00, "price_out": 5.00},
        }
    },
    "xai": {
    "models": {
        "grok-3-mini": {"supplier": "xai", "price_in": 0.30, "price_out": 0.50},
        }
    },
    "google": {
    "models": {
        "gemini-2.5-flash-lite": {"supplier": "google", "price_in": 0.10, "price_out": 0.40},
        }
    },
    "deepseek": {
    "models": {
        "deepseek-chat":     {"supplier": "deepseek", "price_in": 0.27, "price_out": 1.10},
    }
},
}

RUNS = [1, 2, 3]

print("Config loaded.")
print(f"Models : {sum(len(v['models']) for v in VENDORS.values())} across {len(VENDORS)} vendors")
print(f"Suppliers: {set(m['supplier'] for v in VENDORS.values() for m in v['models'].values())}")

Config loaded.
Models : 6 across 5 vendors
Suppliers: {'openai', 'deepseek', 'google', 'xai', 'anthropic'}


In [ ]:
SYSTEM_PROMPT = """Label this product review for a consumer psychology study.

TASK A — EMOTIONS (multi-label, binary 0/1):
Return 1 if clearly present, 0 if not. Only label with explicit evidence.
Joy, Love, Surprise, Pride, Gratitude, Contentment, Relief, Hope, Interest,
Anger, Frustration, Disappointment, Sadness, Fear, Guilt, Disgust

TASK B — EVALUATIONS (single label each):
Trust: Low / Medium / High
Commitment: Low / Medium / High
Recommendation: Would Not Recommend / Neutral / Would Recommend
Sentiment: Negative / Neutral / Positive

Rules: explicit evidence only, middle label if unclear, no skipping eval dims.

Return ONLY this JSON, nothing else:
{
  "emotions": {
    "Joy": 0, "Love": 0, "Surprise": 0, "Pride": 0, "Gratitude": 0,
    "Contentment": 0, "Relief": 0, "Hope": 0, "Interest": 0,
    "Anger": 0, "Frustration": 0, "Disappointment": 0,
    "Sadness": 0, "Fear": 0, "Guilt": 0, "Disgust": 0
  },
  "Trust": "Medium",
  "Commitment": "Medium",
  "Recommendation": "Neutral",
  "Sentiment": "Neutral"
}"""

print("System prompt set. Length:", len(SYSTEM_PROMPT), "chars")

System prompt set. Length: 1011 chars


In [ ]:
# get_api_key() and userdata are defined in the setup cell above.

def parse_response(text):
    """Parse JSON response into emotions dict + eval dims. Adapted from Ringel Cell 17."""
    try:
        text = text.strip()
        # Strip thinking tags (DeepSeek style)
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        # Strip markdown fences
        if "```" in text:
            lines = [l for l in text.split("\n") if not l.strip().startswith("```")]
            text = "\n".join(lines).strip()

        # Find first complete JSON object using brace matching
        start = text.find("{")
        if start == -1:
            return None, None, "No JSON object found"

        depth = 0
        end = start
        for i, char in enumerate(text[start:], start):
            if char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break

        # If braces unbalanced (truncated response), close it
        if depth > 0:
            text = text + "}" * depth

        data = json.loads(text[start:end] if depth == 0 else text[start:])

        # Validate emotions
        emotions = data.get("emotions", {})
        if not isinstance(emotions, dict):
            return None, None, "emotions not a dict"
        for em in EMOTIONS:
            if em not in emotions:
                emotions[em] = 0
            emotions[em] = int(bool(emotions[em]))

        # Validate eval dims
        eval_out = {}
        for dim, valid_levels in EVAL_DIMS.items():
            val = data.get(dim, "")
            if val not in valid_levels:
                eval_out[dim] = valid_levels[len(valid_levels) // 2]
            else:
                eval_out[dim] = val

        return emotions, eval_out, None

    except Exception as e:
        return None, None, str(e)

def compute_cost(model_info, input_tokens, output_tokens):
    return (input_tokens / 1e6) * model_info["price_in"] + \
           (output_tokens / 1e6) * model_info["price_out"]

# ── Per-vendor call functions (adapted from Ringel Cell 15/19) ─────────────────
def call_openai(review_text, model, model_info):
    client = OpenAI(api_key=get_api_key("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": review_text[:2000]}
        ],
        temperature=0,
        max_tokens=300,
        response_format={"type": "json_object"},
    )
    text = resp.choices[0].message.content or ""
    return text, resp.usage.prompt_tokens, resp.usage.completion_tokens

def call_anthropic(review_text, model, model_info):
    time.sleep(0.5)
    client = anthropic.Anthropic(api_key=get_api_key("claude"))
    resp = client.messages.create(
        model=model,
        max_tokens=300,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": review_text[:2000]}],
        temperature=0,
    )
    text = resp.content[0].text if resp.content else ""
    return text, resp.usage.input_tokens, resp.usage.output_tokens

def call_xai(review_text, model, model_info):
    client = OpenAI(
        api_key=get_api_key("xai"),
        base_url="https://api.x.ai/v1",
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": review_text[:2000]}
        ],
        temperature=0,
        max_tokens=300,
    )
    text = resp.choices[0].message.content or ""
    return text, resp.usage.prompt_tokens, resp.usage.completion_tokens

def call_google(review_text, model, model_info):
    client = genai.Client(api_key=get_api_key("gemini"))
    resp = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=review_text[:2000],
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0,
            max_output_tokens=2048,
        )
    )
    text = resp.text or ""
    in_tok  = resp.usage_metadata.prompt_token_count or 0
    out_tok = resp.usage_metadata.candidates_token_count or 0
    return text, in_tok, out_tok

def call_deepseek(review_text, model, model_info):
    client = OpenAI(
        api_key=get_api_key("deepseek"),
        base_url="https://api.deepseek.com/v1",
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": review_text[:2000]}
        ],
        temperature=0,
        max_tokens=300,
    )
    text = resp.choices[0].message.content or ""
    return text, resp.usage.prompt_tokens, resp.usage.completion_tokens

def call_fireworks(review_text, model, model_info):
    client = OpenAI(
        api_key=get_api_key("minimax"),
        base_url="https://api.fireworks.ai/inference/v1"
    )
    fw_model_map = {
        "minimax-m2p7": "accounts/fireworks/models/minimax-m2p7",
    }
    full_model = fw_model_map.get(model, model)
    resp = client.chat.completions.create(
        model=full_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": review_text[:2000]}
        ],
        temperature=0,
        max_tokens=400,
    )
    text = resp.choices[0].message.content or ""
    return text, resp.usage.prompt_tokens, resp.usage.completion_tokens

def call_model(vendor, model, review_text, model_info, retries=3):
    """Route to correct vendor with retry logic. Adapted from Ringel classify_sentence()."""
    wait_times = [3, 6, 15]
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            if vendor == "openai":
                raw, in_tok, out_tok = call_openai(review_text, model, model_info)
            elif vendor == "anthropic":
                raw, in_tok, out_tok = call_anthropic(review_text, model, model_info)
            elif vendor == "xai":
                raw, in_tok, out_tok = call_xai(review_text, model, model_info)
            elif vendor == "google":
                raw, in_tok, out_tok = call_google(review_text, model, model_info)
            elif vendor == "deepseek":
                raw, in_tok, out_tok = call_deepseek(review_text, model, model_info)
            elif vendor == "fireworks":
                raw, in_tok, out_tok = call_fireworks(review_text, model, model_info)
            else:
                return None, None, None, f"Unknown vendor: {vendor}"
            latency = time.perf_counter() - t0

            emotions, eval_dims, err = parse_response(raw)
            if err:
                if attempt < retries - 1:
                    time.sleep(wait_times[attempt])
                    continue
                return None, None, None, f"Parse error: {err}"

            cost = compute_cost(model_info, in_tok, out_tok)
            meta = {
                "input_tokens": in_tok, "output_tokens": out_tok,
                "cost_usd": cost, "latency_sec": latency,
                "attempts": attempt + 1, "error": None
            }
            return emotions, eval_dims, meta, None

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait_times[attempt])
            else:
                return None, None, None, str(e)

print("API functions ready.")

API functions ready.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_model_on_holdout(vendor, model, run=1, max_workers=10):
    """
    Label the full holdout with one model, one run.
    Uses parallel threads for speed. Adapted from Ringel classify_dataframe().
    """
    model_info = VENDORS[vendor]["models"][model]
    model_slug = model.replace("/", "-")
    base_name  = f"holdout_{vendor}_{model_slug}_run{run}"
    run_dir    = f"{HOLDOUT_DIR}/Run{run}"
    os.makedirs(run_dir, exist_ok=True)

    checkpoint_path = f"{CHECKPOINT_DIR}/{base_name}.pkl"
    output_path     = f"{run_dir}/{base_name}.pkl"

    # Resume from checkpoint if exists
    results       = []
    processed_ids = set()
    if os.path.exists(checkpoint_path):
        try:
            ck            = pd.read_pickle(checkpoint_path)
            successful    = ck[ck["error"].isna()]
            results       = successful.to_dict("records")
            processed_ids = set(successful["id"].values)
            print(f"  Resuming: {len(results)} done, {len(ck)-len(results)} errors to retry")
        except Exception as e:
            print(f"  Could not load checkpoint: {e}")

    # Load holdout
    df_holdout       = pd.read_csv(f"{OUTPUT_DIR}/holdout_raw_v3.csv")
    df_holdout["id"] = df_holdout.index

    # Filter out already processed
    df_todo = df_holdout[~df_holdout["id"].isin(processed_ids)].copy()

    print(f"\n{'='*60}")
    print(f"Vendor  : {vendor}")
    print(f"Model   : {model}")
    print(f"Supplier: {model_info['supplier']}")
    print(f"Run     : {run}")
    print(f"Pricing : ${model_info['price_in']}/M in, ${model_info['price_out']}/M out")
    print(f"Reviews : {len(df_holdout)} | Already done: {len(processed_ids)} | To do: {len(df_todo)}")
    print(f"Workers : {max_workers}")
    print(f"{'='*60}")

    start_time = datetime.datetime.now()
    lock       = __import__("threading").Lock()

    def process_row(row):
        emotions, eval_dims, meta, err = call_model(
            vendor, model, row["review_text"], model_info
        )
        if err:
            return {
                "id": row["id"], "review_text": row["review_text"],
                "category": row["category"], "rating": row["rating"],
                **{em: None for em in EMOTIONS},
                **{dim: None for dim in EVAL_DIMS},
                "input_tokens": 0, "output_tokens": 0,
                "cost_usd": 0, "latency_sec": 0,
                "attempts": 3, "error": err
            }
        return {
            "id": row["id"], "review_text": row["review_text"],
            "category": row["category"], "rating": row["rating"],
            **emotions, **eval_dims, **meta
        }

    rows = [row for _, row in df_todo.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_row, row): row["id"] for row in rows}
        completed = 0
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"{model} run{run}"):
            result = future.result()
            with lock:
                results.append(result)
                completed += 1
                # Checkpoint every 100 completions
                if completed % 100 == 0:
                    pd.DataFrame(results).to_pickle(checkpoint_path)

    df_results = pd.DataFrame(results)
    df_results.to_pickle(checkpoint_path)
    df_results.to_pickle(output_path)
    df_results.to_csv(output_path.replace(".pkl", ".csv"), index=False)

    duration   = (datetime.datetime.now() - start_time).total_seconds()
    successful = df_results[df_results["error"].isna()]
    total_cost = successful["cost_usd"].sum()
    print(f"\nDone: {len(successful)}/{len(df_results)} successful")
    print(f"Cost: ${total_cost:.4f} | Runtime: {duration:.0f}s ({duration/60:.1f}m)")
    print(f"Saved: {output_path}")
    return df_results

print("Parallel classification loop ready.")

Parallel classification loop ready.


In [ ]:
def compute_majority_vote(vendor, model, runs=RUNS):
    """
    Majority vote across runs for one model.
    Emotions: present if >50% of runs say present.
    Eval dims: most common label across runs.
    Adapted from Ringel Cell 63/64.
    """
    model_slug = model.replace("/", "-")
    run_dfs    = {}
    for run in runs:
        path = f"{HOLDOUT_DIR}/Run{run}/holdout_{vendor}_{model_slug}_run{run}.pkl"
        if not os.path.exists(path):
            print(f"  Missing: {path}")
            return None
        df           = pd.read_pickle(path)
        run_dfs[run] = df[df["error"].isna()].copy()

    # Align on common ids across all runs
    common_ids = set(run_dfs[runs[0]]["id"])
    for run in runs[1:]:
        common_ids &= set(run_dfs[run]["id"])
    print(f"  {vendor}/{model}: {len(common_ids)} common reviews across {len(runs)} runs")

    results = []
    for row_id in sorted(common_ids):
        rows = {run: run_dfs[run][run_dfs[run]["id"] == row_id].iloc[0]
                for run in runs}

        result = {
            "id":          row_id,
            "review_text": rows[runs[0]]["review_text"],
            "category":    rows[runs[0]]["category"],
            "rating":      rows[runs[0]]["rating"],
        }

        # Majority vote on emotions
        for em in EMOTIONS:
            votes             = sum(int(rows[run][em] or 0) for run in runs)
            result[em]        = 1 if votes > len(runs) / 2 else 0
            result[f"{em}_votes"] = votes

        # Majority vote on eval dims
        for dim, levels in EVAL_DIMS.items():
            level_votes = {lv: 0 for lv in levels}
            for run in runs:
                val = rows[run].get(dim)
                if val in level_votes:
                    level_votes[val] += 1
            result[dim]            = max(level_votes, key=level_votes.get)
            result[f"{dim}_votes"] = level_votes

        result["total_cost_usd"]  = sum(rows[run]["cost_usd"] for run in runs)
        result["avg_latency_sec"] = np.mean([rows[run]["latency_sec"] for run in runs])
        results.append(result)

    df_mv    = pd.DataFrame(results)
    out_path = f"{HOLDOUT_DIR}/majority_vote_{vendor}_{model_slug}.pkl"
    df_mv.to_pickle(out_path)
    df_mv.to_csv(out_path.replace(".pkl", ".csv"), index=False)
    print(f"  Saved: {out_path}")
    return df_mv

print("Computing majority votes...")
mv_results = {}
for vendor, info in VENDORS.items():
    for model in info["models"]:
        key = f"{vendor}_{model}"
        mv  = compute_majority_vote(vendor, model)
        if mv is not None:
            mv_results[key] = mv
print(f"\nDone: {len(mv_results)} majority votes computed")

Computing majority votes...
  openai/gpt-4o-mini: 1441 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_openai_gpt-4o-mini.pkl
  openai/gpt-4.1-mini: 1441 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_openai_gpt-4.1-mini.pkl
  anthropic/claude-haiku-4-5-20251001: 1256 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_anthropic_claude-haiku-4-5-20251001.pkl
  xai/grok-3-mini: 1441 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_xai_grok-3-mini.pkl
  google/gemini-2.5-flash-lite: 1441 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_google_gemini-2.5-flash-lite.pkl
  deepseek/deepseek-chat: 1441 common reviews across 3 runs
  Saved: data/holdout_llm_labels/majority_vote_deepseek_deepseek-chat.pkl

Done: 6 majority votes computed


In [ ]:
def compute_alpha_across_runs(vendor, model, runs=RUNS):
    """
    Krippendorff's alpha per emotion across runs of the same model.
    Measures reproducibility. Adapted from Ringel Cell 60/61.
    """
    model_slug = model.replace("/", "-")
    run_dfs    = {}
    for run in runs:
        path = f"{HOLDOUT_DIR}/Run{run}/holdout_{vendor}_{model_slug}_run{run}.pkl"
        if not os.path.exists(path):
            return None
        df           = pd.read_pickle(path)
        run_dfs[run] = df[df["error"].isna()].set_index("id")

    common_ids = sorted(set.intersection(*[set(run_dfs[r].index) for r in runs]))

    alphas = {}
    for em in EMOTIONS:
        matrix = np.array([
            [int(run_dfs[run].loc[i, em] or 0) for i in common_ids]
            for run in runs
        ])
        try:
            alphas[em] = krippendorff.alpha(
                reliability_data=matrix, level_of_measurement="nominal"
            )
        except:
            alphas[em] = np.nan

    # Overall across all emotions
    overall_matrix = np.hstack([
        np.array([
            [int(run_dfs[run].loc[i, em] or 0) for i in common_ids]
            for run in runs
        ])
        for em in EMOTIONS
    ])
    try:
        alphas["Overall"] = krippendorff.alpha(
            reliability_data=overall_matrix, level_of_measurement="nominal"
        )
    except:
        alphas["Overall"] = np.nan

    return alphas

print("Computing self-consistency alpha (across runs)...")
alpha_results = {}
for vendor, info in VENDORS.items():
    for model in info["models"]:
        key    = f"{vendor}_{model}"
        alphas = compute_alpha_across_runs(vendor, model)
        if alphas:
            alpha_results[key] = alphas
            print(f"\n{key}")
            print(f"  Overall: {alphas['Overall']:.4f}")
            for em in EMOTIONS:
                print(f"  {em:<15}: {alphas[em]:.4f}")

Computing self-consistency alpha (across runs)...

openai_gpt-4o-mini
  Overall: 0.9374
  Joy            : 0.9454
  Love           : 0.9401
  Surprise       : 0.9303
  Pride          : 0.7975
  Gratitude      : 0.9319
  Contentment    : 0.9157
  Relief         : 0.8957
  Hope           : 0.9056
  Interest       : 0.8600
  Anger          : 0.9341
  Frustration    : 0.9467
  Disappointment : 0.9584
  Sadness        : 0.9591
  Fear           : 0.9417
  Guilt          : 0.9678
  Disgust        : 0.9310

openai_gpt-4.1-mini
  Overall: 0.9470
  Joy            : 0.9598
  Love           : 0.9624
  Surprise       : 0.9416
  Pride          : 0.8453
  Gratitude      : 0.9004
  Contentment    : 0.9451
  Relief         : 0.8981
  Hope           : 0.8905
  Interest       : 0.8884
  Anger          : 0.9088
  Frustration    : 0.9232
  Disappointment : 0.9536
  Sadness        : 0.9702
  Fear           : 0.9295
  Guilt          : 0.9075
  Disgust        : 0.9333

anthropic_claude-haiku-4-5-20251001
  Ov

In [ ]:
# ── Clean human labels file ───────────────────────────────────────────────────
df_human_raw = pd.read_csv(f"{OUTPUT_DIR}/final_holdout.csv")

# Drop duplicate Joy column
df_human_raw = df_human_raw.drop(columns=["Joy (0/1).1"], errors="ignore")

# Rename columns to match code expectations
emotion_rename = {
    "Joy (0/1)":            "Joy",
    "Love (0/1) ":          "Love",
    "Surprise (0/1)":       "Surprise",
    "Pride (0/1)":          "Pride",
    "Gratitude (0/1)":      "Gratitude",
    "Contentment (0/1)":    "Contentment",
    "Relief (0/1)":         "Relief",
    "Hope (0/1)":           "Hope",
    "Interest (0/1)":       "Interest",
    "Anger (0/1)":          "Anger",
    "Frustration (0/1)":    "Frustration",
    "Disappointment (0/1)": "Disappointment",
    "Sadness (0/1)":        "Sadness",
    "Fear (0/1)":           "Fear",
    "Guilt (0/1)":          "Guilt",
    "Disgust (0/1)":        "Disgust",
}
eval_rename = {
    "Trust (Low / Medium / High)":                                              "Trust",
    "Commitment (Low / Medium / High)":                                         "Commitment",
    "Recommendation Intent (Would Not Recommend / Neutral / Would Recommend)":  "Recommendation",
    "Sentiment (Negative / Neutral / Positive)":                                "Sentiment",
}
df_human = df_human_raw.rename(columns={**emotion_rename, **eval_rename})

print(f"Human labels loaded: {len(df_human)} reviews")
print(f"Missing emotions  : {[e for e in EMOTIONS if e not in df_human.columns]}")
print(f"Missing eval dims : {[d for d in EVAL_DIMS if d not in df_human.columns]}")

print("\nHuman label distribution (emotions):")
for em in EMOTIONS:
    count = df_human[em].sum()
    pct   = count / len(df_human) * 100
    print(f"  {em:<15}: {int(count)} ({pct:.1f}%)")

# ── Evaluate each model vs human labels ───────────────────────────────────────
def evaluate_vs_human(df_mv, df_human, model_name):
    """
    Compare model majority-vote labels to human labels.
    Adapted from Ringel Cell 66/67 evaluate_multilabel().
    """
    merged = df_mv.merge(
        df_human[["review_text"] + EMOTIONS + list(EVAL_DIMS.keys())],
        on="review_text", suffixes=("_model", "_human")
    )
    print(f"\n{model_name}: {len(merged)} matched reviews")

    results = {}

    # ── Emotion metrics ───────────────────────────────────────────────────────
    y_true = merged[[f"{em}_human" for em in EMOTIONS]].values.astype(int)
    y_pred = merged[[f"{em}_model" for em in EMOTIONS]].values.astype(int)

    per_class = {}
    for i, em in enumerate(EMOTIONS):
        yt, yp = y_true[:, i], y_pred[:, i]
        try:
            auc = roc_auc_score(yt, yp)
        except:
            auc = np.nan
        per_class[em] = {
            "f1":        f1_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score(yt, yp, zero_division=0),
            "mcc":       matthews_corrcoef(yt, yp),
            "auc":       auc,
            "support":   int(yt.sum()),
        }

    results["per_emotion"]   = per_class
    results["emotion_macro"] = {
        "f1_macro":    np.mean([per_class[e]["f1"]  for e in EMOTIONS]),
        "mcc_macro":   np.mean([per_class[e]["mcc"] for e in EMOTIONS]),
        "auc_macro":   np.nanmean([per_class[e]["auc"] for e in EMOTIONS]),
        "f1_micro":    f1_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming":     hamming_loss(y_true, y_pred),
        "exact_match": (y_pred == y_true).all(axis=1).mean(),
    }

    # ── Eval dim metrics ──────────────────────────────────────────────────────
    eval_metrics = {}
    for dim, levels in EVAL_DIMS.items():
        yt = merged[f"{dim}_human"].values
        yp = merged[f"{dim}_model"].values
        eval_metrics[dim] = {
            "f1_macro": f1_score(yt, yp, average="macro", zero_division=0, labels=levels),
            "accuracy": (yt == yp).mean(),
        }
    results["eval_dims"] = eval_metrics

    # ── Krippendorff alpha vs human ───────────────────────────────────────────
    alphas_vs_human = {}
    for em in EMOTIONS:
        matrix = np.array([
            merged[f"{em}_human"].values.astype(int),
            merged[f"{em}_model"].values.astype(int),
        ])
        try:
            alphas_vs_human[em] = krippendorff.alpha(
                reliability_data=matrix, level_of_measurement="nominal"
            )
        except:
            alphas_vs_human[em] = np.nan

    overall_matrix = np.hstack([
        np.array([
            merged[f"{em}_human"].values.astype(int),
            merged[f"{em}_model"].values.astype(int),
        ])
        for em in EMOTIONS
    ])
    try:
        alphas_vs_human["Overall"] = krippendorff.alpha(
            reliability_data=overall_matrix, level_of_measurement="nominal"
        )
    except:
        alphas_vs_human["Overall"] = np.nan

    results["alpha_vs_human"]  = alphas_vs_human
    results["total_cost_usd"]  = df_mv["total_cost_usd"].sum()
    results["avg_latency_sec"] = df_mv["avg_latency_sec"].mean()
    return results

# Run evaluation for all models
eval_results = {}
for key, df_mv in mv_results.items():
    eval_results[key] = evaluate_vs_human(df_mv, df_human, key)

Human labels loaded: 1441 reviews
Missing emotions  : []
Missing eval dims : []

Human label distribution (emotions):
  Joy            : 547 (38.0%)
  Love           : 353 (24.5%)
  Surprise       : 303 (21.0%)
  Pride          : 31 (2.2%)
  Gratitude      : 109 (7.6%)
  Contentment    : 982 (68.1%)
  Relief         : 156 (10.8%)
  Hope           : 63 (4.4%)
  Interest       : 205 (14.2%)
  Anger          : 96 (6.7%)
  Frustration    : 399 (27.7%)
  Disappointment : 569 (39.5%)
  Sadness        : 112 (7.8%)
  Fear           : 118 (8.2%)
  Guilt          : 82 (5.7%)
  Disgust        : 111 (7.7%)

openai_gpt-4o-mini: 1200 matched reviews

openai_gpt-4.1-mini: 1200 matched reviews

anthropic_claude-haiku-4-5-20251001: 1050 matched reviews

xai_grok-3-mini: 1200 matched reviews

google_gemini-2.5-flash-lite: 1200 matched reviews

deepseek_deepseek-chat: 1200 matched reviews


In [ ]:
print("\n" + "="*100)
print("BENCHMARK RESULTS — GenAI vs Human Labels")
print("="*100)

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for model_key, res in eval_results.items():
    em = res["emotion_macro"]
    rows.append({
        "Model":          model_key,
        "Macro F1":       round(em["f1_macro"],    4),
        "Macro MCC":      round(em["mcc_macro"],   4),
        "Macro AUC":      round(em["auc_macro"],   4),
        "Micro F1":       round(em["f1_micro"],    4),
        "Exact Match":    round(em["exact_match"], 4),
        "Hamming Loss":   round(em["hamming"],     4),
        "Alpha vs Human": round(res["alpha_vs_human"]["Overall"], 4),
        "Self Alpha":     round(alpha_results.get(model_key, {}).get("Overall", np.nan), 4),
        "Total Cost $":   round(res["total_cost_usd"],  4),
        "Avg Latency s":  round(res["avg_latency_sec"], 2),
    })

df_summary = pd.DataFrame(rows).sort_values("Macro F1", ascending=False)
print(df_summary.to_string(index=False))

# ── Per-emotion breakdown for best model ──────────────────────────────────────
best_model = df_summary.iloc[0]["Model"]
print(f"\n{'='*100}")
print(f"PER-EMOTION BREAKDOWN — Best Model: {best_model}")
print(f"{'='*100}")
print(f"{'Emotion':<15} {'F1':>7} {'AUC':>7} {'MCC':>7} {'Prec':>7} {'Rec':>7} {'Support':>8} {'Alpha_vs_Human':>15}")
print("-"*85)
for em in EMOTIONS:
    m = eval_results[best_model]["per_emotion"][em]
    a = eval_results[best_model]["alpha_vs_human"][em]
    print(f"{em:<15} {m['f1']:>7.4f} {m['auc']:>7.4f} {m['mcc']:>7.4f} "
          f"{m['precision']:>7.4f} {m['recall']:>7.4f} {m['support']:>8} {a:>15.4f}")

# ── Eval dims breakdown ───────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("EVALUATION DIMENSIONS — Macro F1 & Accuracy vs Human")
print(f"{'='*80}")
print(f"{'Model':<40} {'Dimension':<20} {'Macro F1':>10} {'Accuracy':>10}")
print("-"*80)
for model_key, res in eval_results.items():
    for dim, metrics in res["eval_dims"].items():
        print(f"{model_key:<40} {dim:<20} {metrics['f1_macro']:>10.4f} {metrics['accuracy']:>10.4f}")

# ── Save ──────────────────────────────────────────────────────────────────────
df_summary.to_csv(f"{OUTPUT_DIR}/step7_benchmark_summary.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/step7_benchmark_summary.csv")
print(f"\nBest model: {best_model} (Macro F1 = {df_summary.iloc[0]['Macro F1']})")
print("Use this model for Step 8 to label the full train/test set.")


BENCHMARK RESULTS — GenAI vs Human Labels
                              Model  Macro F1  Macro MCC  Macro AUC  Micro F1  Exact Match  Hamming Loss  Alpha vs Human  Self Alpha  Total Cost $  Avg Latency s
anthropic_claude-haiku-4-5-20251001    0.6237     0.5882     0.8163    0.7457       0.1762        0.1029          0.6812      0.9816        5.5208           3.45
                openai_gpt-4.1-mini    0.6067     0.5721     0.7692    0.7390       0.2117        0.0911          0.6838      0.9470        1.7955          10.28
                 openai_gpt-4o-mini    0.5844     0.5463     0.7495    0.6948       0.1633        0.1020          0.6336      0.9374        0.6700           7.65
             deepseek_deepseek-chat    0.5837     0.5554     0.7362    0.6721       0.1817        0.0997          0.6134      0.9374        1.2645           8.60
       google_gemini-2.5-flash-lite    0.5293     0.5085     0.7117    0.6353       0.1367        0.1094          0.5709      1.0000        0.5098 

In [ ]:
# ── Per-emotion alpha vs human: all models side by side ──────────────────────
print("\n" + "="*120)
print("PER-EMOTION ALPHA VS HUMAN — All Models")
print("="*120)

model_keys = list(eval_results.keys())
model_short = {k: k.split("_")[0] + "/" + k.split("_", 1)[1][:20] for k in model_keys}

# Header
header = f"{'Emotion':<16}" + "".join(f"{model_short[k]:>22}" for k in model_keys) + f"{'BEST MODEL':>25}"
print(header)
print("-" * len(header))

routing_table = {}

for em in EMOTIONS:
    alphas = {k: eval_results[k]["alpha_vs_human"].get(em, np.nan) for k in model_keys}
    best_key = max(alphas, key=lambda k: alphas[k] if not np.isnan(alphas[k]) else -999)
    routing_table[em] = best_key
    row = f"{em:<16}" + "".join(f"{alphas[k]:>22.4f}" for k in model_keys) + f"  → {model_short[best_key]}"
    print(row)

# Also do eval dims
print("\n" + "="*120)
print("PER-EVAL-DIM ACCURACY VS HUMAN — All Models")
print("="*120)
print(f"{'Dimension':<16}" + "".join(f"{model_short[k]:>22}" for k in model_keys) + f"{'BEST MODEL':>25}")
print("-" * len(header))

for dim in EVAL_DIMS:
    accs = {k: eval_results[k]["eval_dims"][dim]["accuracy"] for k in model_keys}
    best_key = max(accs, key=lambda k: accs[k])
    routing_table[dim] = best_key
    row = f"{dim:<16}" + "".join(f"{accs[k]:>22.4f}" for k in model_keys) + f"  → {model_short[best_key]}"
    print(row)

print("\n" + "="*80)
print("ROUTING TABLE (best model per label)")
print("="*80)
for label, model_key in routing_table.items():
    print(f"  {label:<20} → {model_short[model_key]}")

# Save routing table
routing_df = pd.DataFrame([
    {"label": label, "best_model": model_key, "vendor_model": model_key}
    for label, model_key in routing_table.items()
])
routing_df.to_csv(f"{OUTPUT_DIR}/step8_routing_table.csv", index=False)
print(f"\nSaved routing table: {OUTPUT_DIR}/step8_routing_table.csv")

[... earlier output truncated ...]
Sadness                         0.4475                0.3143                0.3278                0.4564                0.2045                0.4560  → xai/grok-3-mini
Fear                            0.6107                0.6358                0.7674                0.5888                0.6552                0.7587  → anthropic/claude-haiku-4-5-202
Guilt                           0.3589                0.4386                0.4091                0.4937                0.1950                0.2708  → xai/grok-3-mini
Disgust                         0.7378                0.6689                0.6477                0.6911                0.5215                0.7210  → openai/gpt-4o-mini

PER-EVAL-DIM ACCURACY VS HUMAN — All Models
Dimension           openai/gpt-4o-mini   openai/gpt-4.1-minianthropic/claude-haiku-4-5-202       xai/grok-3-minigoogle/gemini-2.5-flash-litdeepseek/deepseek-chat               BEST MODEL
-------------------------------------------

In [ ]:
# ── SETTLE THE DEBATE: True alpha of routed ensemble vs Claude-only ────────────
print("\n" + "="*70)
print("ROUTED ENSEMBLE ALPHA vs CLAUDE-ONLY ALPHA")
print("True Krippendorff's alpha computed on assembled label matrices")
print("="*70)

# Step 1: Build the routed ensemble label matrix
# For each label, take the majority-vote labels from the best model
routing = pd.read_csv(f"{OUTPUT_DIR}/step8_routing_table.csv")
routing_dict = dict(zip(routing["label"], routing["best_model"]))

# Merge all models' majority votes with human labels on review_text
# Start from df_human (ground truth) and pull each label from its assigned model
df_ensemble = df_human[["review_text"] + EMOTIONS + list(EVAL_DIMS.keys())].copy()
df_ensemble = df_ensemble.rename(columns={c: f"{c}_human" for c in EMOTIONS + list(EVAL_DIMS.keys())})

for label, best_model_key in routing_dict.items():
    df_mv = mv_results[best_model_key]
    col   = label
    merged = df_ensemble.merge(
        df_mv[["review_text", col]].rename(columns={col: f"{col}_routed"}),
        on="review_text", how="inner"
    )
    df_ensemble = merged

print(f"Reviews matched across all models: {len(df_ensemble)}")

# Step 2: Compute true Krippendorff's alpha for routed ensemble vs human
routed_alphas = {}
for em in EMOTIONS:
    matrix = np.array([
        df_ensemble[f"{em}_human"].values.astype(int),
        df_ensemble[f"{em}_routed"].values.astype(int),
    ])
    try:
        routed_alphas[em] = krippendorff.alpha(reliability_data=matrix, level_of_measurement="nominal")
    except:
        routed_alphas[em] = np.nan

# Overall routed ensemble alpha (joint matrix)
overall_matrix_routed = np.hstack([
    np.array([
        df_ensemble[f"{em}_human"].values.astype(int),
        df_ensemble[f"{em}_routed"].values.astype(int),
    ])
    for em in EMOTIONS
])
routed_overall = krippendorff.alpha(reliability_data=overall_matrix_routed, level_of_measurement="nominal")

# Step 3: Retrieve Claude-only overall alpha (already computed)
claude_key     = "anthropic_claude-haiku-4-5-20251001"
claude_overall = eval_results[claude_key]["alpha_vs_human"]["Overall"]

# Step 4: Print comparison
print(f"\n{'Label':<18} {'Claude-only α':>15} {'Routed ensemble α':>18} {'Winner':>10}")
print("-" * 65)
for em in EMOTIONS:
    claude_a = eval_results[claude_key]["alpha_vs_human"].get(em, np.nan)
    routed_a = routed_alphas.get(em, np.nan)
    winner   = "Routed" if routed_a > claude_a else "Claude"
    print(f"{em:<18} {claude_a:>15.4f} {routed_a:>18.4f} {winner:>10}")

print("-" * 65)
print(f"{'OVERALL (joint)':<18} {claude_overall:>15.4f} {routed_overall:>18.4f}", end="")
winner_overall = "Routed ✓" if routed_overall > claude_overall else "Claude ✓"
print(f" {winner_overall:>10}")

print(f"""
KEY INSIGHT:
The overall alpha is computed on the JOINT label matrix — not the average
of per-label alphas. A routed ensemble can win individual labels but still
score lower overall because mixing models introduces cross-label inconsistency.
Claude's high self-consistency (α=0.9816 across runs) means its labels are
coherent across all 20 dimensions simultaneously, which benefits the joint score.
""")


ROUTED ENSEMBLE ALPHA vs CLAUDE-ONLY ALPHA
True Krippendorff's alpha computed on assembled label matrices
Reviews matched across all models: 1050

Label                Claude-only α  Routed ensemble α     Winner
-----------------------------------------------------------------
Joy                         0.5334             0.6845     Routed
Love                        0.6700             0.6752     Routed
Surprise                    0.5550             0.5550     Claude
Pride                       0.2210             0.2655     Routed
Gratitude                   0.3969             0.4413     Routed
Contentment                 0.8721             0.8721     Claude
Relief                      0.4186             0.5447     Routed
Hope                        0.5075             0.5075     Claude
Interest                    0.4404             0.4947     Routed
Anger                       0.6125             0.6717     Routed
Frustration                 0.6464             0.6531     Routed
Disapp